### Block 0 – Install required packages

In [16]:
!pip install -q openai datasets huggingface_hub scikit-learn tqdm pandas


### Block 1 – Imports and OpenAI client setup

In [17]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfFolder
from openai import OpenAI
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

# OpenAI key
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (sk-...): ")
client = OpenAI()

# Hugging Face token for gated dataset
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf-...): ")


OpenAI API key (sk-...): ··········
Hugging Face token (hf-...): ··········


### Block 2 – Load FiQA SA dataset

In [18]:
DATASET_ID = "TheFinAI/flare-fiqasa"

# Load all splits with the verified token
ds_all = load_dataset(DATASET_ID, token=hf_token)

# Use the official test split (235 examples)
EVAL_SPLIT = "test"
dataset = ds_all[EVAL_SPLIT]

print("Available splits and sizes:")
for split_name in ds_all.keys():
    print(f"  {split_name}: {len(ds_all[split_name])}")

print("\nUsing split:", EVAL_SPLIT)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])

# Hard check to ensure we are really using 235 examples
assert len(dataset) == 235, f"Expected 235 test examples, got {len(dataset)}"


Available splits and sizes:
  train: 750
  test: 235
  valid: 188

Using split: test
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 3 – Define label set and normalization helper

In [19]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map free-form model output to one of the three labels.
    If no label keyword is found, default to 'neutral'.
    """
    text = (raw or "").strip().lower()
    for label in LABELS:
        if label in text:
            return label
    return "neutral"


### Block 4 – GPT-4o sentiment classifier

In [20]:
# Choose the model to evaluate
# Examples: "gpt-4o", "o3-mini", "gpt-5"
MODEL_NAME = "gpt-4o"   # <-- change this to evaluate a different model

INSTRUCTIONS = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog, "
    "classify its overall sentiment from the perspective of an investor as "
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_model(sentence: str) -> str:
    """
    Call the chosen GPT model once and return a normalized 3-way label.
    This uses the 'responses' API and max_output_tokens instead of max_tokens.
    """
    response = client.responses.create(
        model=MODEL_NAME,
        instructions=INSTRUCTIONS,
        input=sentence,
        max_output_tokens=16,
    )
    raw = response.output_text or ""
    return normalize_prediction(raw)


### Block 5 – Run evaluation loop over FiQA SA test split with GPT-4o

In [21]:
y_true = []
y_pred = []

print(f"Evaluating model: {MODEL_NAME} on {len(dataset)} FiQA-SA test examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_model(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: gpt-4o on 235 FiQA-SA test examples...



100%|██████████| 235/235 [03:10<00:00,  1.23it/s]

Total examples: 235
Got predictions for: 235


### Block 6 – Compute metrics and inspect errors

In [22]:
# Overall metrics
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

# Build a DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.7660

Classification report:
              precision    recall  f1-score   support

    negative       0.87      0.88      0.88        76
     neutral       0.23      0.72      0.35        18
    positive       0.98      0.71      0.82       141

    accuracy                           0.77       235
   macro avg       0.69      0.77      0.68       235
weighted avg       0.89      0.77      0.80       235


First 10 predictions:
                                                                                                                            text     gold     pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative negative
                                                                         Legal & General share price: Finance chief to step down negative negative
                                                                 Kingfisher share price slides on cost to